# DenseNet121 on CIFAR-10

这个 Notebook 从 `CIFAR-10` 数据集加载开始，完整展示 `Resize(224x224) -> DenseNet121` 的训练与结构分析流程。

内容包括：
- 数据集预处理与可视化
- `DataLoader` 构建
- 经典 DenseNet121 实现
- Dense Block 与 Transition Layer 解读
- 特征拼接机制分析
- 参数量与尺寸变化分析
- 训练、验证与预测展示

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具用于可视化排版等辅助逻辑
import math
# dataclass 用于统一管理实验配置
from dataclasses import dataclass

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# torchvision 提供常见数据集与图像预处理工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据下载与缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 从 32x32 拉伸到 224x224
    image_size: int = 224
    # 每个 batch 的样本数
    batch_size: int = 32
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 学习率
    lr: float = 1e-3
    # 训练轮数
    epochs: int = 5
    # DenseNet121 的经典 growth rate
    growth_rate: int = 32


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

DenseNet121 通常配合较大尺寸输入一起讨论。这里先把 `CIFAR-10` 从 `32x32` 拉伸到 `224x224`，便于完整观察经典 DenseNet 的多阶段结构。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 训练集：尺寸拉伸、随机翻转、张量化、归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，只保留基础预处理
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 本地没有数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# 类别名称
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于可视化显示
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 展示若干样本，确认输入数据形式
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集打乱顺序，减少模型对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 测试集保持固定顺序即可
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的维度
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. DenseNet121 实现

DenseNet 的核心思想不是残差相加，而是特征拼接。每一层都会接收前面所有层的输出，因此能够显式复用已有特征。DenseNet121 是这一系列中最经典的代表结构之一。

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        # DenseNet-BC 风格：先 1x1 压缩，再 3x3 生成新特征
        inter_channels = growth_rate * 4
        self.layers = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, inter_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(inter_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_channels, growth_rate, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x):
        # 每个 dense layer 只生成 growth_rate 个新通道
        new_features = self.layers(x)
        # DenseNet 的关键：在通道维上把旧特征和新特征拼接起来
        return torch.cat([x, new_features], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super().__init__()
        layers = []
        channels = in_channels
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)


class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 过渡层负责压缩通道并做下采样，控制特征规模
        self.layers = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.layers(x)


class DenseNet121CIFAR10(nn.Module):
    def __init__(self, growth_rate=32, num_classes=10):
        super().__init__()
        # 经典 DenseNet121 使用四个 dense block，层数分别为 6, 12, 24, 16
        block_layers = [6, 12, 24, 16]
        init_channels = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, init_channels, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(init_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        channels = init_channels

        self.block1 = DenseBlock(block_layers[0], channels, growth_rate)
        channels = self.block1.out_channels
        self.trans1 = TransitionLayer(channels, channels // 2)
        channels = channels // 2

        self.block2 = DenseBlock(block_layers[1], channels, growth_rate)
        channels = self.block2.out_channels
        self.trans2 = TransitionLayer(channels, channels // 2)
        channels = channels // 2

        self.block3 = DenseBlock(block_layers[2], channels, growth_rate)
        channels = self.block3.out_channels
        self.trans3 = TransitionLayer(channels, channels // 2)
        channels = channels // 2

        self.block4 = DenseBlock(block_layers[3], channels, growth_rate)
        channels = self.block4.out_channels

        self.norm = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.trans1(x)
        x = self.block2(x)
        x = self.trans2(x)
        x = self.block3(x)
        x = self.trans3(x)
        x = self.block4(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


model = DenseNet121CIFAR10(growth_rate=cfg.growth_rate).to(device)
model

## 5. 逐层尺寸变化分析

In [ ]:
def inspect_feature_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造一个假的输入，只用于观察网络内部张量尺寸变化
    x = torch.randn(input_shape)
    print(f'input   -> {tuple(x.shape)}')

    x = model.stem(x)
    print(f'stem    -> {tuple(x.shape)}')
    x = model.block1(x)
    print(f'block1  -> {tuple(x.shape)}')
    x = model.trans1(x)
    print(f'trans1  -> {tuple(x.shape)}')
    x = model.block2(x)
    print(f'block2  -> {tuple(x.shape)}')
    x = model.trans2(x)
    print(f'trans2  -> {tuple(x.shape)}')
    x = model.block3(x)
    print(f'block3  -> {tuple(x.shape)}')
    x = model.trans3(x)
    print(f'trans3  -> {tuple(x.shape)}')
    x = model.block4(x)
    print(f'block4  -> {tuple(x.shape)}')
    x = model.norm(x)
    x = model.relu(x)
    x = model.avgpool(x)
    print(f'avgpool -> {tuple(x.shape)}')
    x = torch.flatten(x, 1)
    print(f'flatten -> {tuple(x.shape)}')
    x = model.fc(x)
    print(f'fc      -> {tuple(x.shape)}')


inspect_feature_shapes(model.cpu())
model = model.to(device)

## 6. DenseNet121 的关键机制解读

1. 特征拼接而不是相加
   - 每一层都能直接访问前面所有层的输出。

2. 特征复用非常充分
   - 早期层学到的边缘、纹理等信息可以一路传到深层使用。

3. 参数效率较好
   - 每层只新增少量通道，避免重复生成整块特征图。

4. 过渡层负责压缩与下采样
   - 如果只做拼接不压缩，通道数会快速膨胀，因此必须用 `Transition Layer` 控制规模。

5. DenseNet121 为什么经典
   - 它在深度、表达能力和结构清晰度之间取得了很好的平衡，因此成为 DenseNet 家族中最常被引用的代表版本之一。

## 7. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计可训练参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

DenseNet121 虽然层数很多，但参数量不一定像 VGG16 那样暴涨，因为 DenseNet 更强调特征复用而不是在每一层都重新生成大块特征。这也是它非常值得分析的地方。

## 8. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 使用 Adam 便于快速开始实验
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 训练模式
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 把一个 batch 的数据送到目标设备
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 评估模式
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

## 9. 训练主循环

DenseNet121 是经典深层网络，在 `224x224` 输入下训练成本不低。默认这里只提供完整训练流程，不主动执行训练命令。

In [ ]:
# 记录训练和验证指标，便于后面画曲线
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

# 每轮先训练，再评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 绘制损失曲线和准确率曲线
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='train loss')
axes[0].plot(epochs, history['val_loss'], label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='train acc')
axes[1].plot(epochs, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 切换到评估模式，保证推理行为稳定
    model.eval()
    # 取一个 batch 做展示
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    preds = logits.argmax(dim=1)

    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 反归一化后再显示图像
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)